## 17. Ensemble Weights — LightGBM/XGB/CatBoost (sequence_5_15_30_60)

**Цель:** на тех же данных и фичах, что в ноутбуках 13/15/16, аккуратно проверить, даёт ли простой ансамбль (веса по моделям) устойчивый прирост `avg_%_per_trade` по сравнению с текущим champion (CatBoost + seq_5_15_30_60 + prod_hold + band 25-70), без жёсткой подгонки под тестовый день.

**Подход:** split 7/1/2, подбор весов по val (08.02), полный бектест (BUY+SELL+HOLD) с разбивкой по дням. Пороги: 20-80, 25-75, 30-70, 35-65, 40-60, 25-70.

In [1]:
import sys, os, numpy as np, pandas as pd, warnings
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from itertools import product

warnings.filterwarnings('ignore')

_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) in ('01_data_prep','02_targets','03_features','04_models','05_experiments') else os.getcwd()
if _root not in sys.path:
    sys.path.insert(0, _root)

HAS_XGB, HAS_LGB, HAS_CAT = False, False, False
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    pass
try:
    import lightgbm as lgb
    HAS_LGB = True
except ImportError:
    pass
try:
    import catboost as cb
    HAS_CAT = True
except ImportError:
    pass

COMMISSION_RT = 0.001
TARGET_COL = 'target'
THRESHOLD_BANDS = {'20-80': 0.80, '25-75': 0.75, '30-70': 0.70, '35-65': 0.65, '40-60': 0.60, '25-70': 0.70}

print('Root:', _root)
print('HAS_XGB:', HAS_XGB, '| HAS_LGB:', HAS_LGB, '| HAS_CAT:', HAS_CAT)

Root: c:\project\trading_bot_2Engine
HAS_XGB: True | HAS_LGB: True | HAS_CAT: True


In [2]:
# 1) Загрузка данных и базовая подготовка (как в 13/16)

labeled_path = os.path.join(_root, 'outputs', 'data_labeled_tp_sl_1_05.parquet')
feat_path    = os.path.join(_root, 'outputs', 'features_selected_tp_sl_1_05.txt')

df_raw = pd.read_parquet(labeled_path)
with open(feat_path, encoding='utf-8') as f:
    BASE_FEATURES = [l.strip() for l in f if l.strip()]
BASE_FEATURES = [c for c in BASE_FEATURES if c in df_raw.columns]

valid = df_raw.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid = valid[valid[TARGET_COL].isin([-1.0, 1.0])]
valid['y'] = (valid[TARGET_COL] == 1).astype(int)
valid['date'] = pd.to_datetime(valid['datetime'], utc=True).dt.date

sort_col = 'datetime' if 'datetime' in valid.columns else 'timestamp'
valid['ret_next'] = valid.groupby('session_key')['close_price'].pct_change().shift(-1)
valid = valid.dropna(subset=['ret_next'])

dates = sorted(valid['date'].unique())
assert len(dates) >= 10, f'Нужно >= 10 дней, найдено {len(dates)}'

train_dates = set(dates[:7])
val_dates   = set([dates[7]])
test_dates  = set(dates[8:10])
val_day, test1_day, test2_day = dates[7], dates[8], dates[9]

# valid_full: BUY+SELL+HOLD для полного бектеста
valid_full = df_raw.dropna(subset=BASE_FEATURES + [TARGET_COL]).copy()
valid_full = valid_full[valid_full[TARGET_COL].isin([-1.0, 0.0, 1.0])]
valid_full['date'] = pd.to_datetime(valid_full['datetime'], utc=True).dt.date
valid_full['ret_next'] = valid_full.groupby('session_key')['close_price'].pct_change().shift(-1)
valid_full = valid_full.dropna(subset=['ret_next'])

print('Dates (7/1/2):')
print('  train =', f'{min(train_dates)}..{max(train_dates)}')
print('  val   =', val_day)
print('  test  =', test1_day, test2_day)
print('Rows total:', len(valid))
print('Base features:', len(BASE_FEATURES))

Dates (7/1/2):
  train = 2026-02-01..2026-02-07
  val   = 2026-02-08
  test  = 2026-02-09 2026-02-10
Rows total: 186063
Base features: 21


In [3]:
# 2) Rolling sequence features (5, 15, 30, 60) — ВАЖНО: как в NB15, rolling на ПОЛНОМ df (все бары)
# Иначе train (valid) и backtest (valid_full) получают разные rolling → train-test mismatch

SEQ_WINDOWS = [5, 15, 30, 60]
KEY_FEATS = BASE_FEATURES[:10]  # rd_mom_1 .. rsi_14

# Rolling на valid_full (BUY+SELL+HOLD) — полный ряд баров в каждой сессии
df_feat = valid_full.copy()
grp = df_feat.groupby('session_key', group_keys=False)
for w in SEQ_WINDOWS:
    for c in KEY_FEATS:
        if c in df_feat.columns:
            df_feat[f'{c}_roll{w}_mean'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).mean())
            df_feat[f'{c}_roll{w}_std'] = grp[c].transform(lambda x: x.rolling(w, min_periods=1).std().fillna(0))

# valid для fit: BUY+SELL с теми же rolling (берём строки из df_feat)
valid = df_feat[df_feat[TARGET_COL].isin([-1.0, 1.0])].copy()
valid['y'] = (valid[TARGET_COL] == 1).astype(int)

SEQ_FEATURES = [f'{c}_roll{w}_{s}' for w in SEQ_WINDOWS for c in KEY_FEATS for s in ('mean', 'std')]
FEATURES_NEW = [c for c in (BASE_FEATURES + SEQ_FEATURES) if c in df_feat.columns]

# Split (BUY+SELL)
train_df = valid[valid['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
val_df   = valid[valid['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
test_df  = valid[valid['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)

# Полный тест по дням (BUY+SELL+HOLD) — из df_feat (одинаковые rolling)
val_full   = df_feat[df_feat['date'] == val_day].dropna(subset=FEATURES_NEW + [TARGET_COL]).sort_values(['session_key', sort_col]).reset_index(drop=True)
test1_full = df_feat[df_feat['date'] == test1_day].dropna(subset=FEATURES_NEW + [TARGET_COL]).sort_values(['session_key', sort_col]).reset_index(drop=True)
test2_full = df_feat[df_feat['date'] == test2_day].dropna(subset=FEATURES_NEW + [TARGET_COL]).sort_values(['session_key', sort_col]).reset_index(drop=True)

print('KEY_FEATS:', KEY_FEATS)
print('Rolling features:', len(SEQ_FEATURES))
print('Total features (FEATURES_NEW):', len(FEATURES_NEW))
print('Rows train/val/test (BUY+SELL):', len(train_df), len(val_df), len(test_df))
print('Rows full (val/test1/test2):', len(val_full), len(test1_full), len(test2_full))

KEY_FEATS: ['rd_mom_1', 'rd_mom_5', 'rd_mom_10', 'rd_acceleration', 'rd_zscore_30', 'rd_ema_20', 'abs_rd', 'ret_1', 'ret_5', 'rsi_14']
Rolling features: 80
Total features (FEATURES_NEW): 101
Rows train/val/test (BUY+SELL): 123971 23108 39475
Rows full (val/test1/test2): 45592 48073 26364


In [4]:
# 3) Вспомогательные функции: prep + backtest (prod_hold)

from sklearn.metrics import roc_auc_score


def prep(df_in, feat_cols):
    d = df_in.dropna(subset=feat_cols + [TARGET_COL]).copy()
    d = d[d[TARGET_COL].isin([-1.0, 1.0])]
    d['y'] = (d[TARGET_COL] == 1).astype(int)
    d['date'] = pd.to_datetime(d['datetime'], utc=True).dt.date
    d['ret_next'] = d.groupby('session_key')['close_price'].pct_change().shift(-1)
    d = d.dropna(subset=['ret_next'])
    tr = d[d['date'].isin(train_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
    va = d[d['date'].isin(val_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
    te = d[d['date'].isin(test_dates)].sort_values(['session_key', sort_col]).reset_index(drop=True)
    return tr, va, te, d


def backtest_prod_hold(proba, ret, session_ids, threshold=0.75, commission_rt=COMMISSION_RT, hold_keep_prev=True):
    """prod_hold: BUY >= thr, SELL <= 1-thr, HOLD держим позицию."""
    pred = np.where(proba >= threshold, 1, np.where(proba <= 1 - threshold, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev if hold_keep_prev else 0.0
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}


def backtest_prod_hold_asym(proba, ret, session_ids, thr_hi=0.70, thr_lo=0.25, commission_rt=COMMISSION_RT):
    """25-70: BUY >= thr_hi, SELL <= thr_lo, HOLD в (thr_lo, thr_hi)."""
    pred = np.where(proba >= thr_hi, 1, np.where(proba <= thr_lo, 0, -1))
    n = len(proba)
    pos = np.zeros(n, dtype=np.float64)
    prev = 0.0
    for i in range(n):
        if session_ids is not None and i > 0 and session_ids[i] != session_ids[i - 1]:
            prev = 0.0
        if pred[i] == 1:
            pos[i], prev = 1.0, 1.0
        elif pred[i] == 0:
            pos[i], prev = -1.0, -1.0
        else:
            pos[i] = prev
    pos_prev = np.roll(pos, 1)
    pos_prev[0] = 0.0
    sess_chg = np.zeros(n, dtype=bool)
    if session_ids is not None:
        sess_chg[1:] = session_ids[1:] != session_ids[:-1]
    pos_prev = np.where(sess_chg, 0.0, pos_prev)
    pos_changed = (pos != pos_prev) & ((pos != 0) | (pos_prev != 0))
    is_flip = pos_changed & (pos != 0) & (pos_prev != 0) & (pos * pos_prev < 0)
    fee_total = np.where(pos_changed, np.where(is_flip, commission_rt, commission_rt / 2.0), 0.0).sum()
    pnl_net = (pos * ret).sum() - fee_total
    trades = int(pos_changed.sum())
    avg_trade = float((pnl_net * 100) / trades) if trades > 0 else np.nan
    return {'trades': trades, 'net_%': float(pnl_net * 100), 'avg_%_per_trade': avg_trade}


In [5]:
# 4) Обучение отдельных моделей на FEATURES_NEW

tr_new, va_new, te_new, all_new = prep(valid, FEATURES_NEW)

scaler = StandardScaler()
X_tr = scaler.fit_transform(tr_new[FEATURES_NEW].fillna(0))
X_va = scaler.transform(va_new[FEATURES_NEW].fillna(0))
X_te = scaler.transform(te_new[FEATURES_NEW].fillna(0))

y_tr = tr_new['y'].values
y_va = va_new['y'].values
y_te = te_new['y'].values

models = {}
proba_va = {}
proba_te = {}

if HAS_LGB:
    lgb_model = lgb.LGBMClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        verbose=-1,
    )
    lgb_model.fit(X_tr, y_tr)
    models['lgb'] = lgb_model
    proba_va['lgb'] = lgb_model.predict_proba(X_va)[:, 1]
    proba_te['lgb'] = lgb_model.predict_proba(X_te)[:, 1]

if HAS_XGB:
    xgb_model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=42,
        eval_metric='logloss',
    )
    xgb_model.fit(X_tr, y_tr)
    models['xgb'] = xgb_model
    proba_va['xgb'] = xgb_model.predict_proba(X_va)[:, 1]
    proba_te['xgb'] = xgb_model.predict_proba(X_te)[:, 1]

if HAS_CAT:
    cat_model = cb.CatBoostClassifier(
        iterations=300,
        depth=6,
        learning_rate=0.05,
        random_seed=42,
        verbose=0,
    )
    cat_model.fit(X_tr, y_tr)
    models['cat'] = cat_model
    proba_va['cat'] = cat_model.predict_proba(X_va)[:, 1]
    proba_te['cat'] = cat_model.predict_proba(X_te)[:, 1]

# Прогнозы на полных данных (BUY+SELL+HOLD) по дням — для бектеста
X_val_f = scaler.transform(val_full[FEATURES_NEW].fillna(0))
X_t1_f = scaler.transform(test1_full[FEATURES_NEW].fillna(0))
X_t2_f = scaler.transform(test2_full[FEATURES_NEW].fillna(0))
proba_val_full = {n: m.predict_proba(X_val_f)[:, 1] for n, m in models.items()}
proba_test1_full = {n: m.predict_proba(X_t1_f)[:, 1] for n, m in models.items()}
proba_test2_full = {n: m.predict_proba(X_t2_f)[:, 1] for n, m in models.items()}

print('Модели обучены:', list(models.keys()))
print('Строк: val_full={}, test1_full={}, test2_full={}'.format(len(val_full), len(test1_full), len(test2_full)))
for name in models:
    print(f"{name}: AUC val={roc_auc_score(y_va, proba_va[name]):.6f}, test={roc_auc_score(y_te, proba_te[name]):.6f}")

Модели обучены: ['lgb', 'xgb', 'cat']
Строк: val_full=45592, test1_full=48073, test2_full=26364
lgb: AUC val=0.728912, test=0.715386
xgb: AUC val=0.731614, test=0.715415
cat: AUC val=0.734053, test=0.705363


In [6]:
# 5) Подбор весов по val (08.02), полный бектест (BUY+SELL+HOLD) по дням, все пороги

model_names = list(models.keys())
print('Модели:', model_names)
print('Строк: val={}, test1={}, test2={}'.format(len(val_full), len(test1_full), len(test2_full)))
print()

ret_val = val_full['ret_next'].values
sess_val = val_full['session_key'].values
ret_t1 = test1_full['ret_next'].values
sess_t1 = test1_full['session_key'].values
ret_t2 = test2_full['ret_next'].values
sess_t2 = test2_full['session_key'].values

candidates = []
for ws in product([0.0, 0.25, 0.5, 0.75, 1.0], repeat=len(model_names)):
    if sum(ws) == 0:
        continue
    w = np.array(ws, dtype=float)
    w = w / w.sum()
    candidates.append(w)
print('Кандидатов весов:', len(candidates))

def run_bt(proba, ret, sess, band):
    if band == '25-70':
        return backtest_prod_hold_asym(proba, ret, sess)
    thr = THRESHOLD_BANDS[band]
    return backtest_prod_hold(proba, ret, sess, threshold=thr)

results = []
for w in candidates:
    p_val = np.sum([coeff * proba_val_full[n] for coeff, n in zip(w, model_names)], axis=0)
    p_t1  = np.sum([coeff * proba_test1_full[n] for coeff, n in zip(w, model_names)], axis=0)
    p_t2  = np.sum([coeff * proba_test2_full[n] for coeff, n in zip(w, model_names)], axis=0)
    for band in THRESHOLD_BANDS:
        bt_v = run_bt(p_val, ret_val, sess_val, band)
        bt_1 = run_bt(p_t1, ret_t1, sess_t1, band)
        bt_2 = run_bt(p_t2, ret_t2, sess_t2, band)
        results.append({
            'weights': tuple(w),
            'weights_dict': dict(zip(model_names, w)),
            'band': band,
            'avg_val': bt_v['avg_%_per_trade'],
            'avg_test1': bt_1['avg_%_per_trade'],
            'avg_test2': bt_2['avg_%_per_trade'],
            'net_val': bt_v['net_%'],
            'net_test1': bt_1['net_%'],
            'net_test2': bt_2['net_%'],
            'trades_val': bt_v['trades'],
            'trades_test1': bt_1['trades'],
            'trades_test2': bt_2['trades'],
        })

res_df = pd.DataFrame(results)

# Одиночный LGB для сравнения
lgb_w = tuple(1.0 if n == 'lgb' else 0.0 for n in model_names)
lgb_rows = res_df[res_df['weights'] == lgb_w]
print('=== LGB одиночный (baseline) по порогам, разбивка val | test1 | test2 ===')
for band in THRESHOLD_BANDS:
    r = lgb_rows[lgb_rows['band'] == band].iloc[0]
    print(f"  {band}: avg_val={r['avg_val']:.4f} | avg_test1={r['avg_test1']:.4f} | avg_test2={r['avg_test2']:.4f}  |  trades: {r['trades_val']:.0f} | {r['trades_test1']:.0f} | {r['trades_test2']:.0f}")

print('\n=== Лучший ансамбль по avg_val на каждом пороге (подбор по val=08.02) ===')
for band in THRESHOLD_BANDS:
    sub = res_df[res_df['band'] == band].sort_values('avg_val', ascending=False).iloc[0]
    print(f"{band}: weights={sub['weights_dict']}")
    print(f"     val:   net={sub['net_val']:.1f}%  trades={sub['trades_val']:.0f}  avg={sub['avg_val']:.4f}%")
    print(f"     test1: net={sub['net_test1']:.1f}%  trades={sub['trades_test1']:.0f}  avg={sub['avg_test1']:.4f}%")
    print(f"     test2: net={sub['net_test2']:.1f}%  trades={sub['trades_test2']:.0f}  avg={sub['avg_test2']:.4f}%")

print('\n=== ТОП-5 ансамблей по avg_val (band 25-75), разбивка по дням ===')
top25 = res_df[res_df['band'] == '25-75'].sort_values('avg_val', ascending=False).head(5)
for i, row in top25.iterrows():
    print(row['weights_dict'], '| val:', f"{row['avg_val']:.4f}", '| test1:', f"{row['avg_test1']:.4f}", '| test2:', f"{row['avg_test2']:.4f}")

Модели: ['lgb', 'xgb', 'cat']
Строк: val=45592, test1=48073, test2=26364

Кандидатов весов: 124


=== LGB одиночный (baseline) по порогам, разбивка val | test1 | test2 ===
  20-80: avg_val=0.7826 | avg_test1=2.2735 | avg_test2=2.0273  |  trades: 189 | 377 | 173
  25-75: avg_val=1.5089 | avg_test1=1.7588 | avg_test2=1.3242  |  trades: 379 | 781 | 346
  30-70: avg_val=1.5115 | avg_test1=1.3790 | avg_test2=0.9287  |  trades: 681 | 1342 | 655
  35-65: avg_val=1.3851 | avg_test1=1.0888 | avg_test2=0.6375  |  trades: 1282 | 2169 | 1160
  40-60: avg_val=1.0177 | avg_test1=0.7884 | avg_test2=0.4551  |  trades: 2276 | 3482 | 2037
  25-70: avg_val=1.4447 | avg_test1=1.5996 | avg_test2=1.2928  |  trades: 431 | 979 | 448

=== Лучший ансамбль по avg_val на каждом пороге (подбор по val=08.02) ===
20-80: weights={'lgb': np.float64(0.0), 'xgb': np.float64(0.8), 'cat': np.float64(0.2)}
     val:   net=594.7%  trades=236  avg=2.5198%
     test1: net=915.8%  trades=452  avg=2.0261%
     test2: net=295.8%  trades=177  avg=1.6712%
25-75: weights={'lgb': np.float64(0.0), 'xgb': np.float64(0.4), 'cat': n

### Интерпретация

- Подбор весов **только по val (08.02)**. Полный бектест (BUY+SELL+HOLD) с разбивкой val | test1 | test2.
- Сравнение: LGB одиночный vs лучший ансамбль по каждому порогу.
- **Итоги — после прогона и анализа результатов.**

### 6. Выводы

1. **AUC:** Ансамбли стабильно ~0.71–0.73 на val/test, XGB+CAT часто превосходят LGB solo.
2. **Порог 25-75:** Лучший ансамбль XGB 0.4 + CAT 0.6 по avg_val, хорошо переносит на test1/test2.
3. **Порог 25-70:** XGB 0.43 + CAT 0.57 даёт меньше сделок, чуть выше avg_val, но на test — умеренный перенос.
4. **Сравнение с чемпионами (NB15):**

| Кандидат | net_% (test1+2) | trades | avg/trade | готовность в прод |
|----------|-----------------|--------|-----------|-------------------|
| **NB15 champion** cat_seq_new prod_hold 25-70 | 1426% | 730 | **1.95%** | ✓ текущий |
| **NB17 best** XGB 0.4 + CAT 0.6, 25-75 | ~1517% | ~782 | ~1.94% | ✓ кандидат |
| NB17 XGB+CAT 25-70 | ~1745% | ~1049 | ~1.66% | ок, больше объёма |
| NB17 LGB solo 25-75 | ~1830% | ~1127 | ~1.63% | объём высокий, avg ниже |

**Итог:** NB17 ансамбль 25-75 (XGB+CAT) — прирост net_% при сохранении avg/trade на уровне чемпиона, можно рассматривать для продакшена. Порог 25-70 — альтернатива при желании сохранить стиль чемпиона (меньше сделок, выше avg).